In [ ]:
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from datetime import datetime
import pymysql

API_KEY = ""

cities = [
    "Surat","Mumbai","Delhi","Bangalore","Ahmedabad","Chennai","Kolkata","Hyderabad","Pune","Jaipur","Udaipur","Vapi"]

username = "root"
password = "password"
host = "localhost"
database = "testd"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}/{database}"
)

weather_data = []


with engine.connect() as conn:

    table_check = conn.execute(text("""
        SHOW TABLES LIKE 'weather_data'
    """))

    table_exists = table_check.fetchone()


for city in cities:

    current_date = datetime.now().date()


    if table_exists:

        check_query = text(f"""
            SELECT COUNT(*) 
            FROM weather_data
            WHERE Weather_Date = '{current_date}'
            AND City = '{city}'
        """)

        with engine.connect() as conn:

            result = conn.execute(check_query)

            count = result.scalar()

    else:

        count = 0


    if count == 0:

        url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"

        response = requests.get(url)

        data = response.json()

        if "main" in data:

            weather_info = {
                "Weather_Date": current_date,
                "Time": datetime.now().strftime("%H:%M:%S"),
                "City": city,
                "Temperature": data["main"]["temp"],
                "Humidity": data["main"]["humidity"],
                "Wind_Speed": data["wind"]["speed"],
                "Weather": data["weather"][0]["main"]
                
            }

            weather_data.append(weather_info)

            print(city,"data inserted")

        else:

            print("Error for",city)

    else:

        print(city, "data already exists")


df = pd.DataFrame(weather_data)


print(df)


if not df.empty:

    df.to_sql(
        "weather_data",
        con=engine,
        if_exists="append",
        index=False
    )

    print("New Data Inserted Into MySQL")

else:

    print("No New Data To Insert")

Surat data inserted
Mumbai data inserted
Delhi data inserted
Bangalore data inserted
Ahmedabad data inserted
Chennai data inserted
Kolkata data inserted
Hyderabad data inserted
Pune data inserted
Jaipur data inserted
Udaipur data inserted
Vapi data inserted
   Weather_Date      Time       City  Temperature  Humidity  Wind_Speed  \
0    2026-05-31  14:45:06      Surat        33.99        62        6.69   
1    2026-05-31  14:45:06     Mumbai        34.99        52        6.17   
2    2026-05-31  14:45:06      Delhi        33.05        46        4.12   
3    2026-05-31  14:45:06  Bangalore        31.04        54        8.05   
4    2026-05-31  14:45:07  Ahmedabad        37.02        34        5.66   
5    2026-05-31  14:45:07    Chennai        35.11        65        5.66   
6    2026-05-31  14:45:07    Kolkata        34.97        46        3.09   
7    2026-05-31  14:45:07  Hyderabad        37.23        30        5.66   
8    2026-05-31  14:45:07       Pune        32.13        47        

In [3]:

df

""
